# Just look up!

This exercise will let you deal with an API which [documentation](https://docs.openaq.org/using-the-api/quick-start) is a little bit more compicated to navigate than the previous one. We'll call an API to get air quality measures for France. Though the API lets you get data from anywhere in the world we have chosen to focus this study on France, but feel free to start with any country of your choice the series of questions would still hold.

1. Go to [https://docs.openaq.org/](https://docs.openaq.org/) and try a first call to make sure the APi is working properly!

In [4]:
import requests

response = requests.get("https://api.openaq.org/ping")

print("status code:", response, "\n\n")
print("Response data:\n")
response.text

status code: <Response [200]> 


Response data:



'Healthy Connection'

2. Get an **API Key**: Sign up for an account on OpenAQ’s platform (if required for an API key). Once signed in, generate an API key to include in your requests. 
   
**This key is necessary for authentication.**

![](https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M07-Data_Collection/OpenAQAccount.png)

1. Based on the [SwaggerUI](https://api.openaq.org/docs#/), find the specific **endpoint** in the SwaggerUI that allows you to retrieve metadata about countries with air quality data.
2. Once the response is received, parse the JSON object to locate the entry for **France**

In [ ]:
# Define the API countries endpoint

# from dotenv import load_dotenv
# import os
# load_dotenv()
BASE_URL = "https://api.openaq.org/v3"
# Endpoint for countries
endpoint = f"{BASE_URL}/countries"
# Add your API key
#api_key = os.getenv("OPENAQ_API_KEY")
headers = {
    "X-API-Key": "f0bae2aaa30554da6e97b2b7a374de0ae9cb2e6fd8ab63a7ac359ee6c296a6c6"
}

# Optional: Parameters for the request (if needed)
params = {
    "limit": 200  # Fetch up to 200 countries (adjust if necessary)
}

# Make the GET request
response = requests.get(endpoint, headers=headers, params=params)
response.raise_for_status()  # Raise an error for bad responses
countries_data = response.json()["results"]
response.content


b'{"meta":{"name":"openaq-api","website":"/","page":1,"limit":200,"found":144},"results":[{"id":1,"code":"ID","name":"Indonesia","datetimeFirst":"2016-01-30T01:00:00Z","datetimeLast":"2026-01-20T14:00:00Z","parameters":[{"id":1,"name":"pm10","units":"\xc2\xb5g/m\xc2\xb3","displayName":null},{"id":2,"name":"pm25","units":"\xc2\xb5g/m\xc2\xb3","displayName":null},{"id":3,"name":"o3","units":"\xc2\xb5g/m\xc2\xb3","displayName":null},{"id":10,"name":"o3","units":"ppm","displayName":null},{"id":11,"name":"bc","units":"\xc2\xb5g/m\xc2\xb3","displayName":null},{"id":15,"name":"no2","units":"ppb","displayName":null},{"id":19,"name":"pm1","units":"\xc2\xb5g/m\xc2\xb3","displayName":null},{"id":21,"name":"co2","units":"ppm","displayName":null},{"id":22,"name":"wind_direction","units":"deg","displayName":null},{"id":23,"name":"nox","units":"ppb","displayName":null},{"id":24,"name":"no","units":"ppb","displayName":null},{"id":34,"name":"wind_speed","units":"m/s","displayName":null},{"id":98,"name"

In [14]:
# Check the response and extract country details for France
france_data = next((country for country in countries_data if country["code"] == "FR"), None)
print(france_data)

{'id': 22, 'code': 'FR', 'name': 'France', 'datetimeFirst': '2016-11-21T11:00:00Z', 'datetimeLast': '2026-01-20T14:00:00Z', 'parameters': [{'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': None}, {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': None}, {'id': 4, 'name': 'co', 'units': 'µg/m³', 'displayName': None}, {'id': 5, 'name': 'no2', 'units': 'µg/m³', 'displayName': None}, {'id': 6, 'name': 'so2', 'units': 'µg/m³', 'displayName': None}, {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': None}, {'id': 98, 'name': 'relativehumidity', 'units': '%', 'displayName': None}, {'id': 100, 'name': 'temperature', 'units': 'c', 'displayName': None}, {'id': 125, 'name': 'um003', 'units': 'particles/cm³', 'displayName': None}, {'id': 19843, 'name': 'no', 'units': 'µg/m³', 'displayName': None}]}


3. Retrieve all locations in France
- Use a pagination loop to handle **multiple pages** of results.
- Filter the results for **country_code == "FR"** during or after data collection.
- Combine all filtered results into a single DataFrame.

In [17]:
import pandas as pd
locations_url=f"{BASE_URL}/locations"
# API endpoint and headers

# Initialize an empty list to store results from all pages
all_locations = []
# Loop through pages 1 to 20
for page in range(1, 21):  # From page 1 to 20
    # Parameters for each request
    params = {
        "limit": 1000,  # Number of results per page
        "page": page    # Current page number
    }
    
    # Make the GET request
    response = requests.get(locations_url, headers=headers, params=params)
    response.raise_for_status()  # Raise an error for bad responses
    data = response.json()["results"]
    all_locations.extend(data)

# Convert the accumulated results into a DataFrame
locations_df = pd.DataFrame(all_locations)

In [20]:
# Display the result
locations_df.head(1)

,id,name,locality,timezone,country,owner,provider,isMobile,isMonitor,instruments,sensors,coordinates,licenses,bounds,distance,datetimeFirst,datetimeLast
0,3,NMA - Nima,None,Africa/Accra,"{'id': 152, 'code': 'GH', 'name': 'Ghana'}","{'id': 4, 'name': 'Unknown Governmental Organi...","{'id': 209, 'name': 'Dr. Raphael E. Arku and C...",False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 6, 'name': 'pm10 µg/m³', 'parameter': ...","{'latitude': 5.58389, 'longitude': -0.19968}",None,"[-0.19968, 5.58389, -0.19968, 5.58389]",None,None,None


In [ ]:
# Extract 'country_code' from 'country' dictionary and filter for country code "FR"
# Extract country code from nested dictionary
locations_df["country_code"] = locations_df["country"].apply(
    lambda c: c.get("code") if isinstance(c, dict) else None
)

# Filter for France
fr_locations_df = locations_df[locations_df["country_code"] == "FR"]




,id,name,locality,timezone,country,owner,provider,isMobile,isMonitor,instruments,sensors,coordinates,licenses,bounds,distance,datetimeFirst,datetimeLast,country_code
2428,2662,NET-FR068A,ATMO OCCITANIE,Europe/Paris,"{'id': 22, 'code': 'FR', 'name': 'France'}","{'id': 4, 'name': 'Unknown Governmental Organi...","{'id': 180, 'name': 'EEA France'}",False,True,"[{'id': 2, 'name': 'Government Monitor'}]","[{'id': 5561, 'name': 'o3 µg/m³', 'parameter':...","{'latitude': 43.69536599999999, 'longitude': 3...","[{'id': 10, 'name': 'ODC-BY', 'attribution': {...","[3.8008169999999994, 43.69536599999999, 3.8008...",None,"{'utc': '2016-11-21T14:00:00Z', 'local': '2016...","{'utc': '2024-03-04T07:00:00Z', 'local': '2024...",FR


In [ ]:
# Display the result
fr_locations_df.head(1)

4. Fetch and Process Air Quality Sensor Data
- Define the `location_id` as `2672` and construct the `location_url` using the OpenAQ API endpoint `https://api.openaq.org/v3/locations/{location_id}/sensors`.
- Iterate over the `results` key in the JSON object and extract the following fields for each sensor:
    - `id`: Sensor ID
    - `name`: Sensor name
    - `parameter_name`: The measured parameter
    - `units`: Units of the measurement
    - `datetime_first_utc`: Date and time of the first observation
    - `datetime_last_utc`: Date and time of the last observation
    - `coverage_expected_count`: Expected count of observations
    - `coverage_observed_count`: Observed count of observations
    - `latest_value`: The most recent measurement value
    - `latitude`: Sensor's latitude
    - `longitude`: Sensor's longitude
    - `summary_min`: Minimum recorded value
    - `summary_max`: Maximum recorded value
    - `summary_avg`: Average recorded value

In [23]:
# Define the location ID and API URL
# dans swagger editor: https://editor.swagger.io/ après authorize et 
# entrée de l'api key 
#GET /v3/locations/{location_id}/sensors

# Make the GET request to fetch location data

# Extract relevant data dynamically
location_id = 2672
location_url = f"{BASE_URL}/locations/{location_id}/sensors"

response = requests.get(location_url, headers=headers)
response.raise_for_status()

sensors_data = response.json()["results"]

processed_sensors = []

for sensor in sensors_data:
    processed_sensors.append({
        "id": sensor.get("id"),
        "name": sensor.get("name"),
        "parameter_name": sensor.get("parameter", {}).get("name"),
        "units": sensor.get("parameter", {}).get("units"),
        "datetime_first_utc": sensor.get("datetimeFirst", {}).get("utc"),
        "datetime_last_utc": sensor.get("datetimeLast", {}).get("utc"),
        "coverage_expected_count": sensor.get("coverage", {}).get("expectedCount"),
        "coverage_observed_count": sensor.get("coverage", {}).get("observedCount"),
        "latest_value": sensor.get("latest", {}).get("value"),
        "latitude": sensor.get("coordinates", {}).get("latitude"),
        "longitude": sensor.get("coordinates", {}).get("longitude"),
        "summary_min": sensor.get("summary", {}).get("min"),
        "summary_max": sensor.get("summary", {}).get("max"),
        "summary_avg": sensor.get("summary", {}).get("avg"),
    })

# Create DataFrame
sensors_df = pd.DataFrame(processed_sensors)





In [24]:
# Display the results
sensors_df.head(1)

,id,name,parameter_name,units,datetime_first_utc,datetime_last_utc,coverage_expected_count,coverage_observed_count,latest_value,latitude,longitude,summary_min,summary_max,summary_avg
0,5579,pm10 µg/m³,pm10,µg/m³,2016-11-21T12:00:00Z,2026-01-20T02:00:00Z,1,32742,24.1,None,None,-2.5,391.8,14.710621
